In [1]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)
import os

from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [2]:
DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
METHOD = "acme"

In [3]:
from data_reading import read_gbm122_data

per_region_total_df , per_region_per_allele_mt_df, per_region_per_allele_wt_df = read_gbm122_data(DATA_PATH)


Region 1, unique (Transcrpit, Epitope pairs): 13197
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*35:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*40:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*03:04: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*04:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
Region 2, unique (Transcrpit, Epitope pairs): 13273
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 5127
allele HLA-B*35:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 51

/home/denklewer/Tools/anacoda3/envs/acme_2/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2818: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


Region 3, unique (Transcrpit, Epitope pairs): 15170
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*35:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*40:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*03:04: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*04:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
Region 4, unique (Transcrpit, Epitope pairs): 13757
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 4932
allele HLA-B*35:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 49

# 0. Preparations

In [4]:
sample_region_dict = next(iter(per_region_per_allele_wt_df.values()))
sample_allele_df =  next(iter(sample_region_dict.values()))

ALLELES = list(sample_region_dict.keys())
REGIONS = list(per_region_total_df.keys())
ALLELES, REGIONS

(['HLA-B*35:01', 'HLA-C*04:01', 'HLA-A*03:01', 'HLA-B*40:01', 'HLA-C*03:04'],
 [1, 2, 3, 4, 5])

## 0.1. make result dicts

In [5]:
per_region_per_method_per_allele_wt_predictions = dict()
per_region_per_method_per_allele_mt_predictions = dict()
for region in per_region_total_df.keys():
    per_region_per_method_per_allele_wt_predictions[region] = dict()
    per_region_per_method_per_allele_mt_predictions[region] = dict()

# 1. Predict with acme

In [18]:
PREDICTOR_HOME = os.getenv('ACME_HOME')
sys.path.append(PREDICTOR_HOME)
import acme_config


'HLA-C*03:04'

In [55]:
## create tempfiles for acme
TEMPDIR = '../TMP/'
def predictWithACME(per_allele_df):
    supported_alleles = [allele_name  for allele_name in per_allele_df.keys() if "C" not in allele_name ]
    
    
    for allele_name in ALLELES:
        target_allele = normalize_allele_name(allele_name).replace('HLA-', '')
        df = per_allele_df[allele_name].copy()
        df['mhc'] = target_allele
        df[df['peptide'].notnull()][['peptide','mhc']].to_csv("{}acme_data_{}".format(TEMPDIR,allele_name), sep='\t',index=False, header=False)
        
    for allele_name in supported_alleles:
        print(allele_name)
        source_filename = "{}acme_data_{}".format(TEMPDIR,allele_name)
        input_filename = '$ACME_HOME/' + acme_config.path_prediction_input
        result_filename = '$ACME_HOME/' + acme_config.path_prediction_results
        destination_filename = '{}outdir_acme/acme_prediction_{}.txt'.format(TEMPDIR,allele_name)
        !cp $source_filename $input_filename
        !python $ACME_HOME/binding_prediction.py
        !cp $result_filename $destination_filename
        !rm $result_filename
    per_allele_predictions = dict()
    for allele_name in supported_alleles:
        result_df = per_allele_df[allele_name].copy()
        result_df['score'] = np.nan
        result_df['peptide_check'] = np.nan
        prediction_df = pd.read_csv('{}outdir_acme/acme_prediction_{}.txt'.format(TEMPDIR, allele_name), sep='\t', header=None, names=['peptide', 'mhc', 'score'])
        result_df.loc[result_df['peptide'].notnull(), 'score'] = prediction_df['score'].values
        result_df.loc[result_df['peptide'].notnull(), 'peptide_check'] = prediction_df['peptide'].values
        per_allele_predictions[allele_name] = result_df
        

    # add dummy predictions
    for allele_name in ALLELES:
        if allele_name not in supported_alleles:
            print("Method doesn't support {}".format(allele_name))
            result_df = per_allele_df[allele_name].copy()
            result_df['score'] = np.nan
            result_df['peptide_check'] = np.nan
            per_allele_predictions[allele_name] = result_df
    return per_allele_predictions
    

In [56]:
for region in REGIONS:
    print("region {}".format(region))
    print("\t mt")
    per_allele_mt_predictions = predictWithACME(per_region_per_allele_mt_df[region])
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions  
    print("\t wt")
    per_allele_wt_predictions = predictWithACME(per_region_per_allele_wt_df[region])
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions 

region 1
	 mt
HLA-B*35:01
Using TensorFlow backend.
2021-08-12 11:11:57.313562: I tensorflow/core/platform/cpu_feature_guard.cc:137] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX AVX2 FMA
HLA-A*03:01
Using TensorFlow backend.
2021-08-12 11:13:26.472031: I tensorflow/core/platform/cpu_feature_guard.cc:137] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX AVX2 FMA
HLA-B*40:01
Using TensorFlow backend.
2021-08-12 11:14:42.993701: I tensorflow/core/platform/cpu_feature_guard.cc:137] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX AVX2 FMA
Method doesn't support HLA-C*04:01
Method doesn't support HLA-C*03:04
	 wt
HLA-B*35:01
Using TensorFlow backend.
2021-08-12 11:15:59.684880: I tensorflow/core/platform/cpu_feature_guard.cc:137] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX 

In [58]:
per_region_per_method_per_allele_wt_predictions[region][METHOD][ALLELES[-1]]

,allele,length,peptide,source_sequence_name,mhc,score,peptide_check
11202,HLA-C*03:04,8,SAFAGLLL,seq11202,C*03:04,NaN,NaN
11203,HLA-C*03:04,8,FSCHVFHL,seq11203,C*03:04,NaN,NaN
11206,HLA-C*03:04,8,YVAVCRPL,seq11206,C*03:04,NaN,NaN
11207,HLA-C*03:04,8,SAVAVCGL,seq11207,C*03:04,NaN,NaN
11209,HLA-C*03:04,8,MGHLGVVL,seq11209,C*03:04,NaN,NaN
11210,HLA-C*03:04,8,MTQVASAV,seq11210,C*03:04,NaN,NaN
11212,HLA-C*03:04,8,FAGLLLLM,seq11212,C*03:04,NaN,NaN
11213,HLA-C*03:04,8,RVLERLVL,seq11213,C*03:04,NaN,NaN
11214,HLA-C*03:04,8,LAQQGTPV,seq11214,C*03:04,NaN,NaN
11224,HLA-C*03:04,8,CSWPGTPV,seq11224,C*03:04,NaN,NaN


# 2. Join predictions

In [59]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity'],
    'acme':['source_sequence_name','score', ]
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df
    

def getPredictionResult(total_results, region, allele_name, method):
    df = total_results[region][method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df


In [62]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_region_per_allele_result = dict()
    for region in REGIONS:
        print("region", region)
        per_region_per_allele_result[region] = dict()
        methods = list(per_region_per_method_per_allele_mt_predictions[region].keys())
        for allele_name in ALLELES:
            print("allele", allele_name)
            df_source_wt = per_region_per_allele_wt_df[region][allele_name].copy()
            df_source_mt = per_region_per_allele_mt_df[region][allele_name].copy()
            
            for method in methods:
                print("method", method)
                df_method_wt = getPredictionResult(per_region_per_method_per_allele_wt_predictions, region, allele_name, method)
                df_method_mt = getPredictionResult(per_region_per_method_per_allele_mt_predictions, region, allele_name, method)
                df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
                df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
            leave_columns = [ 'source_sequence_name', 'allele', 'length']
            df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
            df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
            # rename id back 

            
            df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
            per_region_per_allele_result[region][allele_name] = df_result
    return per_region_per_allele_result
                
                

              

In [63]:
per_region_per_allele_result = joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions)

('region', 1)
('allele', 'HLA-B*35:01')
('method', 'acme')
('allele', 'HLA-C*04:01')
('method', 'acme')
('allele', 'HLA-A*03:01')
('method', 'acme')
('allele', 'HLA-B*40:01')
('method', 'acme')
('allele', 'HLA-C*03:04')
('method', 'acme')
('region', 2)
('allele', 'HLA-B*35:01')
('method', 'acme')
('allele', 'HLA-C*04:01')
('method', 'acme')
('allele', 'HLA-A*03:01')
('method', 'acme')
('allele', 'HLA-B*40:01')
('method', 'acme')
('allele', 'HLA-C*03:04')
('method', 'acme')
('region', 3)
('allele', 'HLA-B*35:01')
('method', 'acme')
('allele', 'HLA-C*04:01')
('method', 'acme')
('allele', 'HLA-A*03:01')
('method', 'acme')
('allele', 'HLA-B*40:01')
('method', 'acme')
('allele', 'HLA-C*03:04')
('method', 'acme')
('region', 4)
('allele', 'HLA-B*35:01')
('method', 'acme')
('allele', 'HLA-C*04:01')
('method', 'acme')
('allele', 'HLA-A*03:01')
('method', 'acme')
('allele', 'HLA-B*40:01')
('method', 'acme')
('allele', 'HLA-C*03:04')
('method', 'acme')
('region', 5)
('allele', 'HLA-B*35:01')
('me

# 5. For now just write to files

In [65]:
DATA_PATH = "/home/denklewer/Data/gbm122/"

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/{}/'.format(METHOD)

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [66]:
# for decoy_key in DECOY_PREFIXES:

for region in REGIONS:
    for allele_name in ALLELES:
        total_df = per_region_per_allele_result[region][allele_name]
        allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        total_df.to_csv("{}{}_predictions_reg_{}_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,region,allele_name),sep=";", index=False)

In [67]:
total_df

,allele,length,peptide_mt,source_sequence_name,mhc_mt,acme_score_mt,peptide_wt,mhc_wt,acme_score_wt
0,HLA-C*03:04,8,SAFAGFLL,seq11202,C*03:04,NaN,SAFAGLLL,C*03:04,NaN
1,HLA-C*03:04,8,FSFHVFHL,seq11203,C*03:04,NaN,FSCHVFHL,C*03:04,NaN
2,HLA-C*03:04,8,YVAVCHPL,seq11206,C*03:04,NaN,YVAVCRPL,C*03:04,NaN
3,HLA-C*03:04,8,SAMAVCGL,seq11207,C*03:04,NaN,SAVAVCGL,C*03:04,NaN
4,HLA-C*03:04,8,MVHLGVVL,seq11209,C*03:04,NaN,MGHLGVVL,C*03:04,NaN
5,HLA-C*03:04,8,MTQVASAM,seq11210,C*03:04,NaN,MTQVASAV,C*03:04,NaN
6,HLA-C*03:04,8,FAGFLLLM,seq11212,C*03:04,NaN,FAGLLLLM,C*03:04,NaN
7,HLA-C*03:04,8,RVLEHLVL,seq11213,C*03:04,NaN,RVLERLVL,C*03:04,NaN
8,HLA-C*03:04,8,LAQQGAPV,seq11214,C*03:04,NaN,LAQQGTPV,C*03:04,NaN
9,HLA-C*03:04,8,CSWPGAPV,seq11224,C*03:04,NaN,CSWPGTPV,C*03:04,NaN
